# Explainability: Grad-CAM and AAL2 region ranking

Runs the native 3D Grad-CAM (`multimodal_ad.models.gradcam`) on a tiny synthetic volume/model, then the AAL2 region-ranking utility (`multimodal_ad.models.regions`) against the real AAL2 atlas bundled at the repository root (`atlas.nii.gz` + `AAL2_Atlas_Labels.csv`), using a synthetic Grad-CAM heatmap so this stays OASIS-free. Requires the `model` extra: run `just install` (installs it along with everything else), or `uv sync --extra model` for a lighter install.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import zoom

from multimodal_ad.models.architecture import Cnn3DConfig, build_3d_cnn
from multimodal_ad.models.gradcam import (
    last_conv_layer_name,
    make_gradcam_heatmap,
    unwrap_output_activation,
)
from multimodal_ad.models.regions import (
    ATLAS_PLACEMENT,
    HEATMAP_PLACEMENT,
    load_atlas,
    load_region_labels,
    pad_to_frame,
    rank_regions,
)

## Native Grad-CAM on a tiny synthetic volume

Same reduced-filter model shape as `02-tiny-model-workflow.ipynb`. Grad-CAM needs raw logits, not a squashed sigmoid output, hence `unwrap_output_activation`.

In [ ]:
SIZE = 48
rng = np.random.default_rng(1234)

config = Cnn3DConfig(
    width=SIZE, height=SIZE, depth=SIZE, filters=(4, 4, 8, 8), dense_units=16,
    name="tiny-gradcam",
)
model = build_3d_cnn(config)
unwrap_output_activation(model)

layer_name = last_conv_layer_name(model)
volume = rng.random((1, SIZE, SIZE, SIZE, 1)).astype("float32")
heatmap = make_gradcam_heatmap(volume, model, layer_name)
heatmap.shape, float(heatmap.min()), float(heatmap.max())

## Visualize the synthetic input and Grad-CAM heatmap

`volume` is random noise (no anatomy), and `model` is untrained, so this
heatmap has no diagnostic meaning; it only demonstrates the shapes and
mechanics of the native 3D Grad-CAM implementation.


In [ ]:
input_slice = volume[0, :, :, SIZE // 2, 0]
heatmap_slice = heatmap[:, :, heatmap.shape[-1] // 2]

# Grad-CAM operates on a coarser late feature map than the input volume;
# resize with a plain zoom (order=1, linear) to overlay it on the input,
# no registration or anatomical alignment implied.
zoom_factors = tuple(v / h for v, h in zip(volume.shape[1:-1], heatmap.shape, strict=True))
resized_heatmap = zoom(heatmap, zoom_factors, order=1)
resized_heatmap_slice = resized_heatmap[:, :, SIZE // 2]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(input_slice, cmap="gray")
axes[0].set_title("Synthetic input (central slice)")
axes[0].axis("off")

axes[1].imshow(heatmap_slice, cmap="jet")
axes[1].set_title("Grad-CAM heatmap (native resolution)")
axes[1].axis("off")

axes[2].imshow(input_slice, cmap="gray")
axes[2].imshow(resized_heatmap_slice, cmap="jet", alpha=0.5)
axes[2].set_title("Grad-CAM resized + overlaid on input")
axes[2].axis("off")

fig.tight_layout()
plt.show()


## AAL2 region ranking: the bundled real atlas

`rank_regions` needs an AAL2 atlas volume and its region labels. Both are bundled at the repository root: `atlas.nii.gz` (the real `(91, 109, 91)` AAL2 volume) and `AAL2_Atlas_Labels.csv` (region name -> intensity mapping). `load_atlas` and `load_region_labels` load them; `pad_to_frame` places the atlas into the common `(128, 128, 128)` frame the legacy notebook used, at the retained `ATLAS_PLACEMENT` offset.

To keep this notebook OASIS-free, the *heatmap* side stays synthetic: a small, deterministic random array standing in for a real Grad-CAM output, padded into the same common frame at `HEATMAP_PLACEMENT`.

In [ ]:
REPO_ROOT = Path.cwd().parent  # notebook kernels cwd to notebooks/

atlas = load_atlas(REPO_ROOT / "atlas.nii.gz")
atlas_frame = pad_to_frame(atlas, ATLAS_PLACEMENT)
region_labels = load_region_labels(str(REPO_ROOT / "AAL2_Atlas_Labels.csv"))

heatmap_rng = np.random.default_rng(42)
synthetic_heatmap = heatmap_rng.random((128, 128, 50)).astype(np.float64)
heatmap_frame = pad_to_frame(synthetic_heatmap, HEATMAP_PLACEMENT)

ranking = rank_regions(atlas_frame, region_labels, {"synthetic": heatmap_frame})
ranking.head()

## Visualize the bundled real AAL2 atlas and the synthetic heatmap overlay

The atlas slice below is the **real, bundled AAL2 atlas** (region labels,
not a scan). The overlay in the second panel combines it with the
**synthetic, random heatmap** used only to exercise `rank_regions`; the
overlap is illustrative, not a validated anatomical registration (see
`ATLAS_PLACEMENT`/`HEATMAP_PLACEMENT` docstrings in
`multimodal_ad.models.regions`).


In [ ]:
central_index = atlas_frame.shape[-1] // 2

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(atlas_frame[:, :, central_index], cmap="nipy_spectral")
axes[0].set_title("Real AAL2 atlas (padded to common frame)")
axes[0].axis("off")

axes[1].imshow(atlas_frame[:, :, central_index] > 0, cmap="gray")
axes[1].imshow(heatmap_frame[:, :, central_index], cmap="jet", alpha=0.5)
axes[1].set_title("Synthetic heatmap over real atlas outline (illustrative only)")
axes[1].axis("off")

fig.tight_layout()
plt.show()


## Legacy `mean` vs. corrected `region mean`

`rank_regions` reports two different mean columns (see `multimodal_ad.models.regions` module docstring for the full derivation):

- `"{name} mean"`: the **legacy notebook's exact formula** (`exploration.ipynb`), region sum divided by the *entire common-frame voxel count* (here, all `128 * 128 * 128` voxels), not the region's own size. Kept for reproducing the published paper's region tables; it shrinks toward zero for small regions purely from a huge, mostly-zero denominator, not from lower Grad-CAM importance.
- `"{name} region mean"`: the **corrected**, size-comparable mean, region sum divided by the region's *own* voxel count. This is what "mean Grad-CAM in region X" should mean, and is what new analysis should use.

In [ ]:
ranking.sort_values("synthetic region mean", ascending=False)[
    ["part", "synthetic mean", "synthetic region mean"]
].head(10)

## Top regions by corrected `region mean`

Horizontal bar chart of the top 10 regions by `"synthetic region mean"`
(the size-comparable, corrected mean), computed from the synthetic
heatmap above; not a claim about which real brain regions matter for AD
detection.


In [ ]:
top_regions = ranking.sort_values("synthetic region mean", ascending=False).head(10)

plt.figure(figsize=(7, 4))
plt.barh(top_regions["part"][::-1], top_regions["synthetic region mean"][::-1])
plt.xlabel("synthetic region mean")
plt.title("Top 10 regions by corrected region mean (synthetic heatmap)")
plt.tight_layout()
plt.show()


## Next steps

- Legacy region-ranking notebook: `notebooks/legacy/exploration-region-ranking.ipynb`.
- Open ambiguities in the atlas/heatmap spatial alignment are documented in `docs/legacy-notebooks-inventory.md`.